In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install transformers rank-bm25 faiss-cpu sentence-transformers -q

import torch
import torch.nn as nn
import torch.nn.functional as F
import json
import numpy as np
import os
from transformers import AutoTokenizer, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi
import faiss
from tqdm import tqdm

DRIVE_BASE = "/content/drive/MyDrive/medrag"
DATA_PATH = f"{DRIVE_BASE}/pubmedqa_filtered.json"
CHECKPOINT_DIR = f"{DRIVE_BASE}/tuned_lens_checkpoints"
OUTPUT_DIR = f"{DRIVE_BASE}/trial2_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

with open(DATA_PATH, "r") as f:
    corpus = json.load(f)

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"Corpus loaded: {len(corpus)} samples")
print(f"Checkpoints: {len(os.listdir(CHECKPOINT_DIR))} files")

In [ ]:
MODEL_ID = "stanford-crfm/BioMedLM"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.float16,
    attn_implementation="eager"
)
model = model.to("cuda")
model.eval()

for param in model.parameters():
    param.requires_grad = False

n_layers = model.config.n_layer
hidden_size = model.config.n_embd
vocab_size = model.config.vocab_size

print(f"Model loaded: {n_layers} layers, hidden size {hidden_size}, vocab {vocab_size}")

In [ ]:
class TunedLensTranslator(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.translator = nn.Linear(hidden_size, hidden_size, bias=True)
        nn.init.eye_(self.translator.weight)
        nn.init.zeros_(self.translator.bias)

    def forward(self, hidden_state):
        return self.translator(hidden_state.float())

translators = nn.ModuleList([
    TunedLensTranslator(hidden_size) for _ in range(n_layers)
])

for layer_idx in range(n_layers):
    ckpt = torch.load(
        os.path.join(CHECKPOINT_DIR, f"translator_layer_{layer_idx:02d}.pt"),
        map_location="cuda"
    )
    translators[layer_idx].load_state_dict(ckpt["state_dict"])

translators = translators.to("cuda")
translators.eval()
for param in translators.parameters():
    param.requires_grad = False

LN_F_WEIGHT = model.transformer.ln_f.weight.float().detach()
LN_F_BIAS = model.transformer.ln_f.bias.float().detach()
LM_HEAD_WEIGHT = model.lm_head.weight.float().detach()

print(f"Tuned-lens loaded: {n_layers} translators ready")

In [ ]:
documents = []
doc_metadata = []

for sample in corpus:
    for abstract in sample["supporting_abstracts"]:
        documents.append(abstract)
        doc_metadata.append({
            "pubid": sample["pubid"],
            "query": sample["query"],
            "gold_answer": sample["gold_answer"],
            "label": sample["label"]
        })

tokenized_docs = [doc.lower().split() for doc in documents]
bm25 = BM25Okapi(tokenized_docs)

enc_model = SentenceTransformer("NeuML/pubmedbert-base-embeddings")
doc_embeddings = enc_model.encode(documents, batch_size=64,
                                   show_progress_bar=True,
                                   convert_to_numpy=True)
faiss.normalize_L2(doc_embeddings)
index = faiss.IndexFlatIP(doc_embeddings.shape[1])
index.add(doc_embeddings)

print(f"Retriever ready: {index.ntotal} documents indexed")

In [ ]:
def bm25_retrieve(query, k=3):
    scores = bm25.get_scores(query.lower().split())
    top_k = np.argsort(scores)[::-1][:k]
    return [{"abstract": documents[i], "pubid": doc_metadata[i]["pubid"],
             "score": float(scores[i]), "gold_answer": doc_metadata[i]["gold_answer"],
             "label": doc_metadata[i]["label"]} for i in top_k]

def faiss_retrieve(query, k=3):
    qe = enc_model.encode([query], convert_to_numpy=True)
    faiss.normalize_L2(qe)
    scores, indices = index.search(qe, k)
    return [{"abstract": documents[i], "pubid": doc_metadata[i]["pubid"],
             "score": float(s), "gold_answer": doc_metadata[i]["gold_answer"],
             "label": doc_metadata[i]["label"]} for s, i in zip(scores[0], indices[0])]

def hybrid_retrieve(query, k=3, rrf_k=60):
    bm25_results = bm25_retrieve(query, k=k*2)
    faiss_results = faiss_retrieve(query, k=k*2)
    combined = {}
    for rank, r in enumerate(bm25_results):
        combined[r["abstract"]] = {"meta": r, "score": 1 / (rrf_k + rank + 1)}
    for rank, r in enumerate(faiss_results):
        key = r["abstract"]
        if key in combined:
            combined[key]["score"] += 1 / (rrf_k + rank + 1)
        else:
            combined[key] = {"meta": r, "score": 1 / (rrf_k + rank + 1)}
    return [v["meta"] for v in sorted(combined.values(),
            key=lambda x: x["score"], reverse=True)[:k]]

def build_rag_prompt_with_spans(query, k=3):
    results = hybrid_retrieve(query, k=k)
    context_parts = [f"[{i+1}] {r['abstract']}" for i, r in enumerate(results)]
    context = "\n\n".join(context_parts)
    prompt = f"Context:\n{context}\n\nQuestion: {query}\nAnswer:"

    span_index = []
    prefix_tokens = tokenizer("Context:\n", return_tensors="pt")["input_ids"].shape[1]
    tokens_so_far = prefix_tokens

    for i, part in enumerate(context_parts):
        part_len = tokenizer(part, return_tensors="pt")["input_ids"].shape[1]
        sep_len = tokenizer("\n\n", return_tensors="pt")["input_ids"].shape[1]
        span_index.append({
            "doc_rank": i,
            "pubid": results[i]["pubid"],
            "token_start": tokens_so_far,
            "token_end": tokens_so_far + part_len,
            "gold_answer": results[i]["gold_answer"],
            "label": results[i]["label"]
        })
        tokens_so_far += part_len + sep_len

    return prompt, results, span_index

print("Retrieval and prompt functions ready.")

In [ ]:
def project_hidden_with_tuned_lens(hidden_state, layer_idx):
    with torch.no_grad():
        translated = translators[layer_idx](hidden_state.float())
        mean = translated.mean(-1, keepdim=True)
        std = translated.std(-1, keepdim=True)
        normalized = (translated - mean) / (std + 1e-5)
        normed = normalized * LN_F_WEIGHT + LN_F_BIAS
        logits = normed @ LM_HEAD_WEIGHT.T
        probs = torch.softmax(logits, dim=-1)
    return probs

STOPWORD_STRINGS = [
    "the", "a", "an", "is", "are", "was", "were", "be", "been", "being",
    "of", "in", "on", "at", "to", "for", "with", "by", "from", "that",
    "this", "these", "those", "it", "its", "as", "or", "and", "but",
    "if", "than", "then", "so", "yet", "both", "each", "more", "most",
    "other", "some", "such", "also", "into", "through", "during", "has",
    "have", "had", "do", "does", "did", "will", "would", "could", "should",
    "may", "might", "must", "can", "their", "they", "them", "there",
    "which", "who", "whom", "what", "when", "where", "how", "all", "any"
]

STOPWORD_IDS = set()
for word in STOPWORD_STRINGS:
    ids = tokenizer(" " + word, add_special_tokens=False)["input_ids"]
    STOPWORD_IDS.update(ids)
    ids2 = tokenizer(word, add_special_tokens=False)["input_ids"]
    STOPWORD_IDS.update(ids2)

print(f"Stopword token IDs: {len(STOPWORD_IDS)} tokens filtered")
print("Negations preserved: 'not', 'no', 'never', 'without' kept in vocabulary")

def get_content_token_ids(gold_answer):
    all_ids = set(tokenizer(gold_answer, add_special_tokens=False)["input_ids"])
    content_ids = all_ids - STOPWORD_IDS
    return content_ids, len(content_ids)

print("Projection and token filtering functions ready.")

In [ ]:
hook_storage = {"hidden_states": {}}
hooks = []

def make_hook(layer_idx):
    def hook(module, input, output):
        hook_storage["hidden_states"][layer_idx] = output[0].detach()
    return hook

for i, block in enumerate(model.transformer.h):
    hooks.append(block.register_forward_hook(make_hook(i)))

N_SAMPLES = len(corpus)
trial2_results = []
low_content_count = 0

print(f"Running Trial 2 on {N_SAMPLES} samples...\n")

for idx, sample in enumerate(tqdm(corpus[:N_SAMPLES])):
    query = sample["query"]
    gold_answer = sample["gold_answer"]

    rag_prompt, retrieved, span_index = build_rag_prompt_with_spans(query, k=3)

    inputs = tokenizer(rag_prompt, return_tensors="pt",
                      truncation=True, max_length=512).to("cuda")
    seq_len = inputs["input_ids"].shape[1]

    hook_storage["hidden_states"].clear()
    with torch.no_grad():
        outputs = model(**inputs)

    content_token_ids, n_content_tokens = get_content_token_ids(gold_answer)
    low_content = n_content_tokens < 3

    if low_content:
        low_content_count += 1

    prob_mass_trajectory = []
    log_prob_mass_trajectory = []

    for layer_idx in range(n_layers):
        hidden = hook_storage["hidden_states"][layer_idx][0]
        probs = project_hidden_with_tuned_lens(hidden, layer_idx)
        last_token_probs = probs[-1]

        evidence_mass = sum(
            last_token_probs[tid].item()
            for tid in content_token_ids
            if tid < vocab_size
        )

        log_evidence_mass = sum(
            torch.log(last_token_probs[tid] + 1e-10).item()
            for tid in content_token_ids
            if tid < vocab_size
        )

        prob_mass_trajectory.append(float(evidence_mass))
        log_prob_mass_trajectory.append(float(log_evidence_mass))

    trial2_results.append({
        "idx": idx,
        "pubid": sample["pubid"],
        "query": query[:60],
        "label": sample["label"],
        "seq_len": seq_len,
        "n_content_tokens": n_content_tokens,
        "low_content": low_content,
        "prob_mass_trajectory": prob_mass_trajectory,
        "log_prob_mass_trajectory": log_prob_mass_trajectory,
        "mean_prob_mass": float(np.mean(prob_mass_trajectory)),
        "max_prob_mass_layer": int(np.argmax(prob_mass_trajectory)),
        "final_layer_prob_mass": prob_mass_trajectory[-1],
        "final_layer_log_prob_mass": log_prob_mass_trajectory[-1]
    })

    if (idx + 1) % 50 == 0:
        ckpt_path = os.path.join(OUTPUT_DIR, f"trial2_checkpoint_{idx+1}.json")
        with open(ckpt_path, "w") as f:
            json.dump(trial2_results, f)
        torch.cuda.empty_cache()
        tqdm.write(f"Checkpoint saved at sample {idx+1} | "
                   f"Low content samples so far: {low_content_count}")

print(f"\nTrial 2 complete. {len(trial2_results)} samples processed.")
print(f"Low content token samples (<3 content tokens): {low_content_count}")

In [ ]:
print("\n TRIAL 2 SUMMARY BY LABEL \n")
for label in ["yes", "no"]:
    label_samples = [r for r in trial2_results if r["label"] == label]
    high_content = [r for r in label_samples if not r["low_content"]]
    print(f"Label: {label} | n={len(label_samples)} | "
          f"high content: {len(high_content)}")
    print(f"  Mean prob mass:       {np.mean([r['mean_prob_mass'] for r in high_content]):.6f}")
    print(f"  Mean final layer mass:{np.mean([r['final_layer_prob_mass'] for r in high_content]):.6f}")
    print(f"  Mean peak layer:      {np.mean([r['max_prob_mass_layer'] for r in high_content]):.1f}")
    print(f"  Mean log prob mass:   {np.mean([r['final_layer_log_prob_mass'] for r in high_content]):.4f}")
    print()

output_path = os.path.join(OUTPUT_DIR, "trial2_full_results.json")
with open(output_path, "w") as f:
    json.dump({
        "n_samples": len(trial2_results),
        "n_layers": n_layers,
        "low_content_count": low_content_count,
        "summary_by_label": {
            label: {
                "n": sum(1 for r in trial2_results if r["label"] == label),
                "n_high_content": sum(1 for r in trial2_results
                                     if r["label"] == label and not r["low_content"]),
                "mean_prob_mass": float(np.mean([r["mean_prob_mass"] for r in trial2_results
                                                if r["label"] == label and not r["low_content"]])),
                "mean_final_prob_mass": float(np.mean([r["final_layer_prob_mass"]
                                                       for r in trial2_results
                                                       if r["label"] == label
                                                       and not r["low_content"]])),
                "mean_peak_layer": float(np.mean([r["max_prob_mass_layer"]
                                                  for r in trial2_results
                                                  if r["label"] == label
                                                  and not r["low_content"]])),
                "mean_final_log_prob": float(np.mean([r["final_layer_log_prob_mass"]
                                                      for r in trial2_results
                                                      if r["label"] == label
                                                      and not r["low_content"]]))
            } for label in ["yes", "no"]
        },
        "samples": trial2_results
    }, f, indent=2)

for fname in os.listdir(OUTPUT_DIR):
    if fname.startswith("trial2_checkpoint"):
        os.remove(os.path.join(OUTPUT_DIR, fname))

for h in hooks:
    h.remove()

print(f"Results saved to {output_path}")
print("Trial 2 complete")

In [ ]:
from google.colab import runtime
runtime.unassign()